# Phase 3 — The five model variants (A–E) and a timing test for the ablation
Builds Models A–E in TensorFlow/Keras, prints their parameter counts for each configuration, and measures
training speed with the ablation protocol of Wahab et al. (2025): Adam, cross-entropy, batch size 128,
dropout 0.3, early stopping (patience 5), 3-fold cross-validation on a 3,000,000-row stratified subset,
SMOTE on the training folds only. The timing decides whether the full protocol fits the compute budget.

In [ ]:
# %% Cell 0 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# %% Cell 1 — model variants
import os, json, time
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

FEAT = '/content/drive/MyDrive/Thesis/outputs_v3/features_v2/'
CONFIGS = ['config1_merged', 'config2_cicids2017', 'config3_ciciot2023']
N_CLASSES = 8
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

def eca(x):
    """Efficient Channel Attention: GAP over positions, 1D conv (k=3) across channels, sigmoid re-weighting."""
    c = x.shape[-1]
    w = layers.GlobalAveragePooling1D()(x)
    w = layers.Reshape((c, 1))(w)
    w = layers.Conv1D(1, 3, padding='same', use_bias=False, activation='sigmoid')(w)
    w = layers.Reshape((1, c))(w)
    return layers.Multiply()([x, w])

def build(variant, n_features, n_classes=N_CLASSES):
    """A: CNN+GAP | B: +ECA | C: +skip | D: +ECA +skip | E: +2xECA +skip"""
    inp = keras.Input((n_features, 1))
    x = layers.Conv1D(64, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    if variant == 'E':
        x = eca(x)
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    if variant in ('B', 'D', 'E'):
        x = eca(x)
    z = layers.GlobalAveragePooling1D()(x)
    if variant in ('C', 'D', 'E'):
        s = layers.Dense(128, activation='relu')(layers.Flatten()(inp))
        z = layers.Concatenate()([z, s])
    z = layers.Dense(64, activation='relu')(z)
    z = layers.Dropout(0.3)(z)
    out = layers.Dense(n_classes, activation='softmax')(z)
    return keras.Model(inp, out, name=f'Model_{variant}')

def n_trainable(m):
    return int(sum(np.prod(v.shape) for v in m.trainable_weights))

n_feat = {c: len(json.load(open(FEAT + c + '/spec.json'))['features']) for c in CONFIGS}
print('features per config:', n_feat)
print(f"{'model':8}" + ''.join(f'{c:>22}' for c in CONFIGS))
for v in 'ABCDE':
    print(f'Model {v} ' + ''.join(f'{n_trainable(build(v, n_feat[c])):>22,}' for c in CONFIGS))

In [ ]:
# %% Cell 2 — timing test (Model D, Config 1, batch 128) and the size of the SMOTE-balanced training folds
from sklearn.model_selection import train_test_split
X = np.load(FEAT + 'config1_merged/X_train.npy', mmap_mode='r')
y = np.load(FEAT + 'config1_merged/y_train.npy')
idx, _ = train_test_split(np.arange(len(y)), train_size=3_000_000, stratify=y, random_state=42)
ys = y[idx]
fold_train = int(len(idx) * 2 / 3)
counts = np.bincount(ys, minlength=N_CLASSES) * 2 / 3
present = counts > 0
print('rows per class in one training fold (2/3 of 3M):', dict(zip(range(N_CLASSES), counts.astype(int))))
print(f'training fold: {fold_train:,} rows | after full SMOTE balancing: {int(counts.max() * present.sum()):,} rows')

m = build('D', n_feat['config1_merged'])
m.compile(optimizer=keras.optimizers.Adam(1e-4), loss='sparse_categorical_crossentropy')
xs = np.asarray(X[np.sort(idx[:128 * 400])])[..., None]
m.fit(xs[:128 * 50], ys[:128 * 50], batch_size=128, epochs=1, verbose=0)          # warm-up
t0 = time.time()
m.fit(xs, ys[:len(xs)], batch_size=128, epochs=1, verbose=0)
sec_per_step = (time.time() - t0) / 400
steps_plain = fold_train / 128
steps_smote = counts.max() * present.sum() / 128
print(f'{sec_per_step*1000:.1f} ms per step')
print(f'one epoch on a training fold: {steps_plain*sec_per_step/60:.1f} min without SMOTE, '
      f'{steps_smote*sec_per_step/60:.1f} min with SMOTE')